# Анализ данных интернет-магазина

In [ ]:
!pip install psycopg2-binary pandas matplotlib

In [ ]:
import psycopg2
import pandas as pd
import os

conn = psycopg2.connect(
    host="postgres",
    database=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD")
)


query = "SELECT * FROM orders"
df = pd.read_sql(query, conn)
df.head()


## Подготовка данных

In [ ]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['total_price'] = df['price'] * df['quantity']

## Общая статистика магазина

In [ ]:
total_orders = len(df)
total_revenue = df['total_price'].sum()

print(f"Total orders: {total_orders}")
print(f"Total revenue: {total_revenue:.2f}")

## Выручка по категориям

In [ ]:
revenue_by_category = (
    df.groupby('category')['total_price']
      .sum()
      .sort_values(ascending=False)
)

revenue_by_category

In [ ]:
import matplotlib.pyplot as plt

revenue_by_category.plot(kind='bar', figsize=(8,5))
plt.title("Revenue by Category")
plt.ylabel("Revenue")
plt.xlabel("Category")
plt.tight_layout()
plt.show()

## Количество заказов по категориям

In [ ]:
orders_by_category = df['category'].value_counts()
orders_by_category

In [ ]:
orders_by_category.plot(kind='bar', figsize=(8,5))
plt.title("Number of Orders by Category")
plt.ylabel("Orders count")
plt.xlabel("Category")
plt.tight_layout()
plt.show()

## Средний чек по категориям

In [ ]:
avg_order_value = (
    df.groupby('category')['total_price']
      .mean()
      .sort_values(ascending=False)
)

avg_order_value

## Динамика заказов во времени

In [ ]:
orders_over_time = (
    df.set_index('created_at')
      .resample('1min')
      .size()
)

orders_over_time.head()

In [ ]:
orders_over_time.plot(figsize=(10,5))
plt.title("Orders Over Time (per minute)")
plt.ylabel("Orders count")
plt.xlabel("Time")
plt.tight_layout()
plt.show()